<a href="https://colab.research.google.com/github/minyi-k03/Large-Language-Model-LLM-/blob/Fine-Tuning/openai_text_generation_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenAI 텍스트 생성(Text Generation) API 살펴보기

## API Reference : https://platform.openai.com/docs/api-reference/chat/create

In [1]:
!pip install openai tiktoken

In [2]:
!pip show openai

Name: openai
Version: 2.14.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: 


# 1. seed 파라미터를 이용해서 재현가능한 텍스트 생성 결과 만들기

## Reference : https://cookbook.openai.com/examples/deterministic_outputs_with_the_seed_parameter

## OpenAI API Key 설정

In [1]:
from openai import AsyncOpenAI

# API 키를 설정하고 클라이언트 생성
client = AsyncOpenAI(api_key="Input Your API Key")

In [2]:
import asyncio
import pprint
import difflib
from IPython.display import display, HTML

GPT_MODEL = "gpt-4o-mini"

In [15]:
# 이 함수는 GPT로부터 응답 결과를 받아옵니다.
async def get_chat_response(system_message: str, user_request: str, seed: int = None):
    try:
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_request},
        ]

        # [수정 1] 비동기(Async) 함수이므로 반드시 await를 붙여야 합니다.
        response = await client.chat.completions.create(
            model=GPT_MODEL,
            messages=messages,
            seed=seed,
            max_tokens=200,
            temperature=0.7,
        )

        response_content = response.choices[0].message.content
        system_fingerprint = response.system_fingerprint

        # [수정 2] 토큰 수는 계산할 필요 없이 바로 속성으로 가져올 수 있습니다.
        prompt_tokens = response.usage.prompt_tokens
        completion_tokens = response.usage.completion_tokens

        table = f"""
        <table>
        <tr><th>Response</th><td>{response_content}</td></tr>
        <tr><th>System Fingerprint</th><td>{system_fingerprint}</td></tr>
        <tr><th>Number of prompt tokens</th><td>{prompt_tokens}</td></tr>
        <tr><th>Number of completion tokens</th><td>{completion_tokens}</td></tr>
        </table>
        """
        display(HTML(table))

        return response_content

    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [16]:
def compare_responses(previous_response: str, response: str):
    if previous_response is None:
        previous_response = ""
    if response is None:
        response = ""

    d = difflib.Differ()
    diff = d.compare(previous_response.splitlines(), response.splitlines())

    diff_table = "<table>"
    diff_exists = False

    for line in diff:
        if line.startswith("- "):
            diff_table += f"<tr style='color: red;'><td>{line}</td></tr>"
            diff_exists = True
        elif line.startswith("+ "):
            diff_table += f"<tr style='color: green;'><td>{line}</td></tr>"
            diff_exists = True
        else:
            diff_table += f"<tr><td>{line}</td></tr>"

    diff_table += "</table>"

    if diff_exists:
        display(HTML(diff_table))
    else:
        print("No differences found.")

## seed 파라미터 설정 없이 비교하기

In [17]:
topic = "가을"
system_message = "You are a helpful assistant that generates short poems."
user_request = f"{topic}에 관한 짧은 시를 작성해줘."

previous_response = await get_chat_response(
    system_message=system_message, user_request=user_request
)

response = await get_chat_response(
    system_message=system_message, user_request=user_request
)

# 이 함수는 두 응답을 비교하고 차이점을 표로 표시합니다.
# 삭제된 부분은 빨간색으로, 추가된 부분은 초록색으로 강조됩니다.
# 차이점이 없는 경우 "차이점이 발견되지 않았습니다."라고 출력합니다.
compare_responses(previous_response, response)

Response,"가을 바람에 노래하는 나뭇잎, 황금빛으로 물든 세상 속, 차가운 공기, 따스한 햇살, 가슴 속 깊이 스며드는 향기. 걷다 보면 들리는 고요한 속삭임, 자연이 그리는 아름다운 그림, 추억의 조각들 함께 나누며, 가을의 품에 안겨 그리움."
System Fingerprint,fp_c4585b5b9c
Number of prompt tokens,34
Number of completion tokens,101


Response,"가을 바람에 나뭇잎 춤추고, 황금빛 햇살이 땅을 감싸네. 가슴 속 깊이 스며드는 향기, 추억의 조각들이 하나둘 피어나. 하늘은 높고 푸르른데, 구름은 솜사탕처럼 둥글고. 걷는 길목마다 사라지는 여름, 가을의 속삭임에 마음이 설레네."
System Fingerprint,fp_c4585b5b9c
Number of prompt tokens,34
Number of completion tokens,108


"+ 가을 바람에 나뭇잎 춤추고,"
+ 황금빛 햇살이 땅을 감싸네.
"- 가을 바람에 노래하는 나뭇잎,"
"- 황금빛으로 물든 세상 속,"
"- 차가운 공기, 따스한 햇살,"
- 가슴 속 깊이 스며드는 향기.
? ^
"+ 가슴 속 깊이 스며드는 향기,"
? ^
+ 추억의 조각들이 하나둘 피어나.
""


## seed 파라미터 설정후 비교하기

In [19]:
#시드 값 고정 후 결과값 비교
#시드값을 고정한다 하더라도 무조건 같은 결과값이 나오지는 않는
SEED = 123
response = await get_chat_response(
    system_message=system_message, seed=SEED, user_request=user_request
)
previous_response = response
response = await get_chat_response(
    system_message=system_message, seed=SEED, user_request=user_request
)

compare_responses(previous_response, response)

Response,"가을 바람 속에 단풍잎이 춤추고, 햇살은 부드럽게 온 세상을 감싸네. 서늘한 공기 속에 추억이 깃들고, 가슴 속 깊이 따스한 사랑이 피어나. 걷는 길마다 황금빛 물결이 흐르고, 가을의 속삭임에 마음이 편안해지네."
System Fingerprint,fp_c4585b5b9c
Number of prompt tokens,34
Number of completion tokens,99


Response,"가을 바람 속에 단풍잎이 춤추고, 하늘은 깊어진 푸름, 황금빛 햇살이 스며든다. 차가운 공기 속에 따스한 차 한 잔, 마음도 함께 녹아내리며 추억을 담아낸다. 가을은 고요히 시간을 속삭여, 새로운 시작을 알리는 끝과 시작의 계절."
System Fingerprint,fp_c4585b5b9c
Number of prompt tokens,34
Number of completion tokens,107


가을 바람 속에
"단풍잎이 춤추고,"
- 햇살은 부드럽게
- 온 세상을 감싸네.
"+ 하늘은 깊어진 푸름,"
+ 황금빛 햇살이 스며든다.
""
- 서늘한 공기 속에
"- 추억이 깃들고,"
- 가슴 속 깊이
- 따스한 사랑이 피어나.


# 2. 토큰 개수 세기

## Reference : https://platform.openai.com/docs/guides/text-generation/managing-tokens

In [20]:
import tiktoken

In [21]:
#미리 토큰 개수 세
def num_tokens_from_messages(messages, model="gpt-4o-mini"):
  """Returns the number of tokens used by a list of messages."""
  try:
      encoding = tiktoken.encoding_for_model(model)
  except KeyError:
      encoding = tiktoken.get_encoding("cl100k_base")
  if model == "gpt-4o-mini":  # note: future models may deviate from this
      num_tokens = 0
      for message in messages:
          num_tokens += 4  # every message follows <im_start>{role/name}\n{content}<im_end>\n
          for key, value in message.items():
              num_tokens += len(encoding.encode(value))
              if key == "name":  # if there's a name, the role is omitted
                  num_tokens += -1  # role is always required and always 1 token
      num_tokens += 2  # every reply is primed with <im_start>assistant
      return num_tokens
  else:
      raise NotImplementedError(f"""num_tokens_from_messages() is not presently implemented for model {model}.
      See https://github.com/openai/openai-python/blob/main/chatml.md for information on how messages are converted to tokens.""")

In [22]:
messages = [
  {"role": "system", "content": "You are a helpful, pattern-following assistant that translates corporate jargon into plain English."},
  {"role": "system", "name":"example_user", "content": "New synergies will help drive top-line growth."},
  {"role": "system", "name": "example_assistant", "content": "Things working well together will increase revenue."},
  {"role": "system", "name":"example_user", "content": "Let's circle back when we have more bandwidth to touch base on opportunities for increased leverage."},
  {"role": "system", "name": "example_assistant", "content": "Let's talk later when we're less busy about how to do better."},
  {"role": "user", "content": "This late pivot means we don't have time to boil the ocean for the client deliverable."},
]

model = "gpt-4o-mini"

print(f"{num_tokens_from_messages(messages, model)} prompt tokens counted.")
# Should show ~126 total_tokens

121 prompt tokens counted.


In [25]:
# example token count from the OpenAI API
response = await client.chat.completions.create(
  model=model,
  messages=messages,
  temperature=0,
)

print(f'{response.usage.prompt_tokens} prompt tokens used.')

124 prompt tokens used.


# 3. JSON mode 사용해보기

In [27]:
response = await client.chat.completions.create(
  model="gpt-4o-mini",
  #미리 타입을 json 형태로 강제 지정한
  response_format={ "type": "json_object" },
  messages=[
    {"role": "system", "content": "You are a helpful assistant designed to output JSON."},
    {"role": "user", "content": "Who won the world series in 2020?"}
  ]
)
print(response.choices[0].message.content)

{
  "year": 2020,
  "world_series_winner": "Los Angeles Dodgers"
}


In [28]:
response = await client.chat.completions.create(
  model="gpt-4o-mini",
  response_format={ "type": "json_object" },
  messages=[
    {"role": "system", "content": "너는 JSON 출력을 만드는 조수야. 아래 내용의 감정상태가 '긍정'인지 '부정'인지 판단해줘"},
    {"role": "user", "content": "가격이 착하고 디자인이 예쁩니다"}
  ]
)
print(response.choices[0].message.content)

{
  "감정상태": "긍정"
}


In [29]:
response = await client.chat.completions.create(
  model="gpt-4o-mini",
  response_format={ "type": "json_object" },
  messages=[
    {"role": "system", "content": "너는 JSON 출력을 만드는 조수야. 아래 내용의 감정상태가 '긍정'인지 '부정'인지 판단해줘"},
    {"role": "user", "content": "금액이 저렴해도 기대를 했었는데 그 값어치밖에 안되는군요 그만큼만 입겠습니다"}
  ]
)
print(response.choices[0].message.content)

{
  "emotion": "부정"
}


# 4. Temperature 값 변경해보기

## temperature : 설정가능범위(0.0~2.0, 기본값 1.0), 낮을수록 더 정확한 답변, 높을수록 더 다양성 있는 답변을 생성

In [30]:
response = await client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": "너는 도움이 되는 조수야."},
    {"role": "user", "content": "푸른 하늘에 OO이 떠있다. OO에 들어갈 단어를 10개 추천해줘"}
  ]
)
print(response.choices[0].message.content)

물론입니다! "푸른 하늘에 OO이 떠있다."의 OO에 들어갈 단어를 10개 추천해드리겠습니다.

1. 구름
2. 새
3. 비행기
4. 해
5. 별
6. 풍선
7. 드론
8. 비행선
9. 태양
10. 나비

원하는 문맥에 맞는 단어를 선택해 보세요!


In [31]:
response = await client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": "너는 도움이 되는 조수야."},
    {"role": "user", "content": "푸른 하늘에 OO이 떠있다. OO에 들어갈 단어를 10개 추천해줘"}
  ],
  temperature=0.0 #Temperature을 지정해서 값을 다양하거나 정확하게 바꿀 수 있다
)
print(response.choices[0].message.content)

물론입니다! "푸른 하늘에 OO이 떠있다." 문장에 들어갈 수 있는 단어 10개를 추천해드릴게요.

1. 구름
2. 새
3. 비행기
4. 태양
5. 달
6. 별
7. 풍선
8. 드론
9. 헬리콥터
10. 나비

이 단어들이 문장에 잘 어울리길 바랍니다!


In [ ]:
response = await client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": "너는 도움이 되는 조수야."},
    {"role": "user", "content": "푸른 하늘에 OO이 떠있다. OO에 들어갈 단어를 10개 추천해줘"}
  ],
  temperature=2.0
)
print(response.choices[0].message.content)

# 5. frequency_penalty & presence_penalty 값 변경해보기

**presence_penalty** : -2.0과 2.0 사이의 숫자입니다. (기본값 0.0) 양수 값은 지금까지의 텍스트에 나타나는 **새로운 토큰을 기반으로 벌칙을 부여**하여 모델이 새로운 주제에 대해 이야기할 가능성을 높입니다.
양수일 경우 최대한 다양성 있는 답변을 하고 음수일 수록 똑같거나 비슷한 단어가 나오게끔 한다. 한번이라도 등장한 단어가 자주 등장하는지 아닌지에 대한 것

**frequency_penalty** : -2.0과 2.0 사이의 숫자입니다. (기본값 0.0) 양수 값은 현재까지 텍스트에서의 **존재 빈도를 기준으로 새 토큰에 대해 벌칙을 부여**하며, 모델이 동일한 문장을 그대로 반복할 가능성을 감소시킵니다.
한번 이상 등장한 단어가 얼마나 자주 반복되는지에 대해 교정하는 파라미터







In [3]:
response = await client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": "너는 도움이 되는 조수야."},
    {"role": "user", "content": "가을 하면 생각나는 단어를 20개 추천해줘"}
  ]
)
print(response.choices[0].message.content)

가을 하면 떠오르는 단어 20개는 다음과 같습니다:

1. 단풍
2. 추수
3. 수확
4. 밤
5. 고구마
6. 감
7. 청명한 하늘
8. 바람
9. 축제
10. 황금색
11. 코스모스
12. 따뜻한 차
13. 이동
14. 갈대
15. 스웨터
16. 해질녘
17. 수수께끼
18. 맛있는 음식
19. 캠핑
20. 나무잎

가을의 풍성함과 아름다움을 잘 나타내는 단어들이에요!


In [4]:
response = await client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": "너는 도움이 되는 조수야."},
    {"role": "user", "content": "가을 하면 생각나는 단어를 20개 추천해줘"}
  ],
  presence_penalty=-2.0 #Presence_Penalty를 음수값으로 지정하여 동일 단어가 얼마나 자주 등장하는지 확인한
)
print(response.choices[0].message.content)

가을 하면 생각나는 단어 20개를 추천해드릴게요!

1. 단풍
2. 추수
3. 감귤
4. 호박
5. 사과
6. 바람
7. 수확
8. 갈대
9. 저녁노을
10. 송편
11. 가을비
12. 뒷산
13. 커피
14. 따뜻한
15. 추워짐
16. 고백
17. 가을산행
18. 캠핑
19. 축제
20. 고백

가을의 매력을 느낄 수 있는 단어들이에요!


In [5]:
response = await client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": "너는 도움이 되는 조수야."},
    {"role": "user", "content": "가을 하면 생각나는 단어를 20개 추천해줘"}
  ],
  presence_penalty=2.0 #Presence_Penalty를 2.0으로 지정해서 최대한 다양한 답변을 유도한
)
print(response.choices[0].message.content)

가을 하면 생각나는 단어 20개를 추천해드릴게요!

1. 단풍
2. 수확
3. 감귤
4. 밤
5. 햇살
6. 삭막함
7. 찬바람
8. 서리
9. 고백
10. 따뜻한 음료
11. 캠핑
12. 코스모스
13. 땅콩
14. 할로윈
15. 추수감사절
16. 가벼운 외투
17. 책읽기
18. 공원 산책
19. 불변의 아름다움
20. 풍선

이 단어들이 가을의 분위기를 잘 담고 있기를 바랍니다!


In [ ]:
response = await client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "system", "content": "너는 도움이 되는 조수야."},
    {"role": "user", "content": "가을 하면 생각나는 단어를 20개 추천해줘"}
  ],
  frequency_penalty=-2.0 #Frequency_Penalty를 음수로 설정한 경우 한번 등장한 단어를 여러번 등장하게끔 강제한
)
print(response.choices[0].message.content)

In [ ]:
response = await client.chat.completions.create(
  model="gpt-3.5-turbo-1106",
  messages=[
    {"role": "system", "content": "너는 도움이 되는 조수야."},
    {"role": "user", "content": "가을 하면 생각나는 단어를 20개 추천해줘"}
  ],
  frequency_penalty=2.0 #Frequency_Penalty의 값을 높게 설정하여 중복되는 단어는 줄지만 이상한 단어를 조합해서 생성한다
)
print(response.choices[0].message.content)

좋아요, 여기 가을과 연관된 20가지 단어에 대한 추천입니다:

1. 단풍
2. 감자
3. 산소
4. 청명
5. 열매
6. 시원한 바람 
7. 걷기 
8. 학교 
9 . 왁스 베리 그래나딘 ,
10 . 수박,
11 . 병충해 관리,
12 . 참외,
13 . 슬레이트 빛깔 ,
14 굴뚝 ,   
15 미 서터니ть ,   
16 펄잡운공주의 이야기     
17 악세사케     
18 맥문갱 여사 경       
19 선장    
20 백합 식물

여러분들이 좋아하는 가을의 특유의 분위기와 관련된 다른 단어가 있으면 추가해 주세요!
